# Customer Intelligence: RFM Segmentation

This notebook extends the retail analysis with **Recency, Frequency, and Monetary (RFM)** customer segmentation. The goal is to identify customer groups that can support retention and loyalty decisions.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_excel('../data/cleaned/sample_-_superstore.xlsx')
df['order_date'] = pd.to_datetime(df['order_date'])
print('Rows:', len(df))
print('Customers:', df['customer_id'].nunique())

Rows: 10194
Customers: 804


## 1. Build RFM metrics

- **Recency:** days since the customer's latest purchase
- **Frequency:** number of distinct orders
- **Monetary:** total sales value

In [2]:
snapshot_date = df['order_date'].max() + pd.Timedelta(days=1)

rfm = (df.groupby(['customer_id','customer_name'])
         .agg(
             last_order=('order_date','max'),
             frequency=('order_id','nunique'),
             monetary=('sales','sum'),
             total_profit=('profit','sum')
         )
         .reset_index())

rfm['recency'] = (snapshot_date - rfm['last_order']).dt.days
rfm = rfm[['customer_id','customer_name','recency','frequency','monetary','total_profit']]
rfm.head()

,customer_id,customer_name,recency,frequency,monetary,total_profit
0,AA-10315,Alex Avila,185,5,5563.560,-362.8825
1,AA-10375,Allen Armold,20,9,1056.390,277.3824
2,AA-10480,Andrew Allen,260,4,1790.512,435.8274
3,AA-10645,Anna Andreadi,56,6,5086.935,857.8033
4,AB-10015,Aaron Bergman,416,3,886.156,129.3465


## 2. Create RFM scores

Customers are scored from 1–5 using quantiles. Higher frequency/monetary scores are better, while lower recency is better.

In [3]:
rfm['r_score'] = pd.qcut(rfm['recency'].rank(method='first'), 5, labels=[5,4,3,2,1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['rfm_score'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']
rfm.sort_values('rfm_score', ascending=False).head(10)

,customer_id,customer_name,recency,frequency,monetary,total_profit,r_score,f_score,m_score,rfm_score
694,SE-20110,Sanjit Engle,10,11,12209.4380,2650.6769,5,5,5,15
35,AI-10855,Arianne Irving,14,10,4375.7860,867.7271,5,5,5,15
713,SM-20950,Suzanne McNair,26,14,5695.7900,610.1142,5,5,5,15
276,EP-13915,Emily Phan,13,17,5478.0608,144.9578,5,5,5,15
315,GZ-14470,Gary Zandusky,8,9,4355.1500,1087.7539,5,5,5,15
329,HM-14860,Harry Marie,3,10,8236.7648,2437.9836,5,5,5,15
82,BF-11170,Ben Ferrer,17,13,8430.1440,1980.3261,5,5,5,15
798,WB-21850,William Brown,21,13,6327.2680,692.3908,5,5,5,15
460,LC-16885,Lena Creighton,17,12,7663.1260,1288.3469,5,5,5,15
524,MH-18115,Mick Hernandez,6,9,5503.0930,170.9746,5,5,5,15


## 3. Customer segments

The segments are intentionally simple and explainable for a business audience.

In [4]:
def segment_customer(row):
    if row['rfm_score'] >= 13 and row['r_score'] >= 4:
        return 'Champions'
    if row['rfm_score'] >= 10 and row['f_score'] >= 3:
        return 'Loyal Customers'
    if row['r_score'] >= 4:
        return 'Recent Customers'
    if row['r_score'] <= 2 and row['f_score'] >= 3:
        return 'At-Risk Customers'
    return 'Occasional Customers'

rfm['segment'] = rfm.apply(segment_customer, axis=1)
segment_summary = (rfm.groupby('segment')
                     .agg(customers=('customer_id','count'), sales=('monetary','sum'), profit=('total_profit','sum'))
                     .sort_values('sales', ascending=False)
                     .reset_index())
segment_summary

,segment,customers,sales,profit
0,Loyal Customers,243,923569.0385,118709.1404
1,Champions,114,600561.9627,78728.1920
2,Occasional Customers,257,458703.2972,45918.0089
3,At-Risk Customers,92,180580.5433,24368.4946
4,Recent Customers,98,163119.5126,24572.9787


In [ ]:
plt.figure(figsize=(9,5))
plt.bar(segment_summary['segment'], segment_summary['customers'])
plt.title('Customers by RFM Segment')
plt.xlabel('Segment')
plt.ylabel('Number of Customers')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 4. Business questions

1. Which segment contains the most customers?
2. Which segment contributes the most sales and profit?
3. How many customers are potentially at risk?
4. Which high-value customers should be prioritized for retention?

In [5]:
at_risk = rfm[rfm['segment']=='At-Risk Customers'].sort_values('monetary', ascending=False)
print('At-risk customers:', len(at_risk))
print('Top at-risk customers by sales:')
display(at_risk.head(10))

champions = rfm[rfm['segment']=='Champions'].sort_values('monetary', ascending=False)
print('Top champions:')
display(champions.head(10))

At-risk customers: 92
Top at-risk customers by sales:


,customer_id,customer_name,recency,frequency,monetary,total_profit,r_score,f_score,m_score,rfm_score,segment
654,RH-19510,Rick Huthwaite,468,6,6979.1800,1289.4535,1,3,5,9,At-Risk Customers
787,VD-21670,Valerie Dominguez,582,6,6442.2540,1617.7874,1,3,5,9,At-Risk Customers
206,DH-13075,Dave Hallsten,278,6,4932.8720,1193.7404,1,3,5,9,At-Risk Customers
625,PW-19240,Pierre Wener,465,7,3922.4150,1290.3540,1,4,4,9,At-Risk Customers
197,DB-13555,Dorothy Badders,108,7,3908.8028,109.3270,2,3,4,9,At-Risk Customers
282,FC-14335,Fred Chung,185,7,3889.3660,714.2926,2,3,4,9,At-Risk Customers
779,TS-21655,Trudy Schmidt,239,5,3368.0940,220.9163,1,3,4,8,At-Risk Customers
650,RF-19345,Randy Ferguson,391,8,3309.1500,633.7072,1,4,4,9,At-Risk Customers
667,RR-19525,Rick Reed,171,6,3302.2620,721.6826,2,3,4,9,At-Risk Customers
542,MP-17470,Mark Packer,719,7,3206.1320,600.2873,1,4,4,9,At-Risk Customers


Top champions:


,customer_id,customer_name,recency,frequency,monetary,total_profit,r_score,f_score,m_score,rfm_score,segment
6,AB-10105,Adrian Barton,42,10,14473.5710,5444.8055,4,5,5,14,Champions
443,KL-16645,Ken Lonsdale,48,12,14175.2290,806.8550,4,5,5,14,Champions
694,SE-20110,Sanjit Engle,10,11,12209.4380,2650.6769,5,5,5,15,Champions
312,GT-14710,Greg Tran,37,11,11820.1200,2163.4269,4,5,5,14,Champions
506,ME-17320,Maria Etezadi,43,10,10663.7280,1859.4695,4,5,5,14,Champions
361,JE-15715,Joe Elijah,47,12,10461.4590,1443.1774,4,5,5,14,Champions
393,JL-15835,John Lee,22,11,9799.9230,228.9070,5,5,5,15,Champions
419,JW-15220,Jane Waco,43,10,8846.3540,2530.5094,4,5,5,14,Champions
358,JD-16150,Justin Deggeller,42,8,8828.0305,1619.5199,4,4,5,13,Champions
608,PK-19075,Pete Kriz,10,12,8646.9340,2038.2676,5,5,5,15,Champions


## 5. Export customer segmentation

The output can be loaded into Power BI to create a customer-segment page.

In [6]:
rfm.to_csv('../data/cleaned/customer_rfm_segments.csv', index=False)
print('Saved: data/cleaned/customer_rfm_segments.csv')

Saved: data/cleaned/customer_rfm_segments.csv
